In [1]:
import torch
from einops import einsum, rearrange

In [2]:
torch.randint(0, 12, size=(12,))

tensor([11,  2,  8, 11,  0,  8, 11,  8, 11,  9,  9, 10])

In [3]:
logits = torch.tensor([0.2, 0.3, -0.2, 0.4, 0.55, 0.8], dtype=torch.float32)

torch.softmax(0.001 * logits, -1)

tensor([0.1666, 0.1667, 0.1666, 0.1667, 0.1667, 0.1667])

In [4]:
def get_angle(position: int, k: int, embedding_dim: int, theta: float):
    return position / pow(theta, (2 * k - 1) / embedding_dim)


In [38]:
d_k = 64
position = 10
theta = 10000
max_seq_len = 12

positions = torch.arange(1, max_seq_len + 1).unsqueeze(-1)

k_s = torch.arange(1, (d_k / 2) + 1)
positions, positions.shape, k_s

(tensor([[ 1],
         [ 2],
         [ 3],
         [ 4],
         [ 5],
         [ 6],
         [ 7],
         [ 8],
         [ 9],
         [10],
         [11],
         [12]]),
 torch.Size([12, 1]),
 tensor([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13., 14.,
         15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26., 27., 28.,
         29., 30., 31., 32.]))

In [41]:
exponent = ((2 * k_s) - 1) / d_k
exponent = exponent.unsqueeze(0)
exponent, exponent.shape

(tensor([[0.0156, 0.0469, 0.0781, 0.1094, 0.1406, 0.1719, 0.2031, 0.2344, 0.2656,
          0.2969, 0.3281, 0.3594, 0.3906, 0.4219, 0.4531, 0.4844, 0.5156, 0.5469,
          0.5781, 0.6094, 0.6406, 0.6719, 0.7031, 0.7344, 0.7656, 0.7969, 0.8281,
          0.8594, 0.8906, 0.9219, 0.9531, 0.9844]]),
 torch.Size([1, 32]))

In [42]:
denominator = torch.pow(theta, exponent)

denominator, denominator.shape, positions.shape

(tensor([[1.1548e+00, 1.5399e+00, 2.0535e+00, 2.7384e+00, 3.6517e+00, 4.8697e+00,
          6.4938e+00, 8.6596e+00, 1.1548e+01, 1.5399e+01, 2.0535e+01, 2.7384e+01,
          3.6517e+01, 4.8697e+01, 6.4938e+01, 8.6596e+01, 1.1548e+02, 1.5399e+02,
          2.0535e+02, 2.7384e+02, 3.6517e+02, 4.8697e+02, 6.4938e+02, 8.6596e+02,
          1.1548e+03, 1.5399e+03, 2.0535e+03, 2.7384e+03, 3.6517e+03, 4.8697e+03,
          6.4938e+03, 8.6596e+03]]),
 torch.Size([1, 32]),
 torch.Size([12, 1]))

In [43]:
position / denominator

tensor([[8.6596e+00, 6.4938e+00, 4.8697e+00, 3.6517e+00, 2.7384e+00, 2.0535e+00,
         1.5399e+00, 1.1548e+00, 8.6596e-01, 6.4938e-01, 4.8697e-01, 3.6517e-01,
         2.7384e-01, 2.0535e-01, 1.5399e-01, 1.1548e-01, 8.6596e-02, 6.4938e-02,
         4.8697e-02, 3.6517e-02, 2.7384e-02, 2.0535e-02, 1.5399e-02, 1.1548e-02,
         8.6596e-03, 6.4938e-03, 4.8697e-03, 3.6517e-03, 2.7384e-03, 2.0535e-03,
         1.5399e-03, 1.1548e-03]])

In [44]:
angles = (positions / denominator).unsqueeze(-1)

angles, angles.shape

(tensor([[[8.6596e-01],
          [6.4938e-01],
          [4.8697e-01],
          [3.6517e-01],
          [2.7384e-01],
          [2.0535e-01],
          [1.5399e-01],
          [1.1548e-01],
          [8.6596e-02],
          [6.4938e-02],
          [4.8697e-02],
          [3.6517e-02],
          [2.7384e-02],
          [2.0535e-02],
          [1.5399e-02],
          [1.1548e-02],
          [8.6596e-03],
          [6.4938e-03],
          [4.8697e-03],
          [3.6517e-03],
          [2.7384e-03],
          [2.0535e-03],
          [1.5399e-03],
          [1.1548e-03],
          [8.6596e-04],
          [6.4938e-04],
          [4.8697e-04],
          [3.6517e-04],
          [2.7384e-04],
          [2.0535e-04],
          [1.5399e-04],
          [1.1548e-04]],
 
         [[1.7319e+00],
          [1.2988e+00],
          [9.7394e-01],
          [7.3035e-01],
          [5.4768e-01],
          [4.1071e-01],
          [3.0799e-01],
          [2.3096e-01],
          [1.7319e-01],
          [1.

In [ ]:
angles[0]

(tensor([[8.6596e-01],
         [6.4938e-01],
         [4.8697e-01],
         [3.6517e-01],
         [2.7384e-01],
         [2.0535e-01],
         [1.5399e-01],
         [1.1548e-01],
         [8.6596e-02],
         [6.4938e-02],
         [4.8697e-02],
         [3.6517e-02],
         [2.7384e-02],
         [2.0535e-02],
         [1.5399e-02],
         [1.1548e-02],
         [8.6596e-03],
         [6.4938e-03],
         [4.8697e-03],
         [3.6517e-03],
         [2.7384e-03],
         [2.0535e-03],
         [1.5399e-03],
         [1.1548e-03],
         [8.6596e-04],
         [6.4938e-04],
         [4.8697e-04],
         [3.6517e-04],
         [2.7384e-04],
         [2.0535e-04],
         [1.5399e-04],
         [1.1548e-04]]),
 torch.Size([1, 32]))

In [10]:
cos = torch.cos(angles)
sin = torch.sin(angles)
min_sin = -sin
cos.shape, sin.shape

(torch.Size([10, 10, 1]), torch.Size([10, 10, 1]))

In [11]:
concat = torch.concat(
    (cos, -sin, sin, cos),
    dim=-1,
)

In [12]:
concat = rearrange(concat, "... (b1 b2) -> ... b1 b2", b1=2)

concat.shape

torch.Size([10, 10, 2, 2])

In [13]:
cos[0], sin[0], concat[0]

(tensor([[ 0.4339],
         [ 0.1576],
         [-0.2060],
         [-0.6194],
         [-0.9482],
         [-0.9185],
         [-0.2431],
         [ 0.7901],
         [ 0.6994],
         [-0.8716]]),
 tensor([[ 0.9010],
         [ 0.9875],
         [ 0.9786],
         [ 0.7851],
         [ 0.3176],
         [-0.3954],
         [-0.9700],
         [-0.6129],
         [ 0.7148],
         [ 0.4902]]),
 tensor([[[ 0.4339, -0.9010],
          [ 0.9010,  0.4339]],
 
         [[ 0.1576, -0.9875],
          [ 0.9875,  0.1576]],
 
         [[-0.2060, -0.9786],
          [ 0.9786, -0.2060]],
 
         [[-0.6194, -0.7851],
          [ 0.7851, -0.6194]],
 
         [[-0.9482, -0.3176],
          [ 0.3176, -0.9482]],
 
         [[-0.9185,  0.3954],
          [-0.3954, -0.9185]],
 
         [[-0.2431,  0.9700],
          [-0.9700, -0.2431]],
 
         [[ 0.7901,  0.6129],
          [-0.6129,  0.7901]],
 
         [[ 0.6994, -0.7148],
          [ 0.7148,  0.6994]],
 
         [[-0.8716, -0.4902],

In [14]:
concat.shape

torch.Size([10, 10, 2, 2])

In [15]:
torch.manual_seed(1234)

x = torch.rand((4, max_seq_len, 10, 2))
token_positions = torch.randint(0, max_seq_len, size=(4, max_seq_len))
indices = torch.arange(4)
rearranged_x = x[:, token_positions][:, 0]  # x[:, token_positions][:, 0]
x.shape, token_positions.shape, rearranged_x.shape, token_positions

(torch.Size([4, 10, 10, 2]),
 torch.Size([4, 10]),
 torch.Size([4, 10, 10, 2]),
 tensor([[6, 0, 8, 1, 2, 7, 4, 9, 6, 4],
         [6, 4, 3, 0, 1, 4, 9, 3, 7, 0],
         [6, 6, 1, 6, 6, 2, 7, 9, 4, 1],
         [2, 1, 1, 7, 4, 1, 7, 9, 7, 4]]))

In [16]:
B, S, *rest = x.shape  # here rest = (10, 2)

idx = token_positions.unsqueeze(-1).unsqueeze(-1).expand(-1, -1, *rest)
reordered = torch.gather(x, dim=1, index=idx)

idx.shape, idx

(torch.Size([4, 10, 10, 2]),
 tensor([[[[6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6],
           [6, 6]],
 
          [[0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0],
           [0, 0]],
 
          [[8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8],
           [8, 8]],
 
          [[1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1],
           [1, 1]],
 
          [[2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2],
           [2, 2]],
 
      

In [17]:
*rest, seq_len, d_k = x.shape  # here rest = (10, 2)

idx = token_positions.unsqueeze(-1).unsqueeze(-1).expand(*rest, seq_len, d_k)
reordered = torch.gather(x, dim=1, index=idx)

idx.shape, token_positions[0], idx[0]

(torch.Size([4, 10, 10, 2]),
 tensor([6, 0, 8, 1, 2, 7, 4, 9, 6, 4]),
 tensor([[[6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6],
          [6, 6]],
 
         [[0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0],
          [0, 0]],
 
         [[8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8],
          [8, 8]],
 
         [[1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1],
          [1, 1]],
 
         [[2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2],
          [2, 2]],
 
         [[7, 7

In [18]:
x[3, 6], reordered[3, 0], x[0, 6].shape, x[:, token_positions][0, 0].shape

(tensor([[0.4783, 0.4637],
         [0.7184, 0.8120],
         [0.9437, 0.3520],
         [0.3704, 0.1080],
         [0.2461, 0.4065],
         [0.2440, 0.5123],
         [0.3927, 0.7875],
         [0.0633, 0.4899],
         [0.3109, 0.5337],
         [0.2299, 0.0076]]),
 tensor([[0.2662, 0.1171],
         [0.0766, 0.3196],
         [0.6247, 0.4895],
         [0.5172, 0.7099],
         [0.6429, 0.8361],
         [0.3802, 0.6147],
         [0.4950, 0.3284],
         [0.1644, 0.3614],
         [0.0268, 0.5593],
         [0.1026, 0.3261]]),
 torch.Size([10, 2]),
 torch.Size([10, 10, 2]))

In [19]:
reordered.shape, concat.shape

(torch.Size([4, 10, 10, 2]), torch.Size([10, 10, 2, 2]))

In [22]:
rot_matrices = concat

rot_matrices.shape, reordered.shape

(torch.Size([10, 10, 2, 2]), torch.Size([4, 10, 10, 2]))

In [30]:
block_diag = torch.block_diag(*rot_matrices[0].unbind())

block_diag[2:4, 2:4], rot_matrices[0][1]

(tensor([[ 0.1576, -0.9875],
         [ 0.9875,  0.1576]]),
 tensor([[ 0.1576, -0.9875],
         [ 0.9875,  0.1576]]))

In [ ]:
full_block_diag = torch.concat(
    [
        torch.block_diag(*rot_matrices[i].unbind()).unsqueeze(0)
        for i in range(rot_matrices.shape[0])
    ],
    dim=0,
)

In [35]:
full_block_diag.shape

torch.Size([10, 20, 20])

In [66]:
k_s = torch.arange(0, (d_k / 2))

(2 * k_s) / d_k

exponent = (2 * k_s) / d_k
denominator = torch.pow(theta, exponent)
positions = torch.arange(0, max_seq_len).unsqueeze(-1)
angles = positions / denominator
angles

tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.0000e+00, 7.4989e-01, 5.6234e-01, 4.2170e-01, 3.1623e-01, 2.3714e-01,
         1.7783e-01, 1.3335e-01, 1.0000e-01, 7.4989e-02, 5.6234e-02, 4.2170e-02,
         3.1623e-02, 2.3714e-02, 1.7783e-02, 1.3335e-02, 1.0000e-02, 7.4989e-03,
         5.6234e-03, 4.2170e-03, 3.1623e-03, 2.3714e-03, 1.7783e-03, 1.3335e-03,
         1.0000e-03, 7.4989e-04, 5.6234e-04, 4.2170e-04, 3.1623e-04, 2.3714e-04,
         1.7783e-04, 1.3335e-04],
        [2.0000e+00, 1.4998e+00, 1.1247e+00, 8.4339e-01, 6.3246e-01, 4.7427e-01,
         3.5566e-01, 2.6670e-01, 2.0000e-

In [ ]:
base = theta

angles = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))

seq_idx = torch.arange(seq_len, device=x.device).float().to(x.device)

idx_theta = torch.einsum("n,d->nd", seq_idx, angles)

idx_theta


tensor([[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00,
         0.0000e+00, 0.0000e+00],
        [1.0000e+00, 7.4989e-01, 5.6234e-01, 4.2170e-01, 3.1623e-01, 2.3714e-01,
         1.7783e-01, 1.3335e-01, 1.0000e-01, 7.4989e-02, 5.6234e-02, 4.2170e-02,
         3.1623e-02, 2.3714e-02, 1.7783e-02, 1.3335e-02, 1.0000e-02, 7.4989e-03,
         5.6234e-03, 4.2170e-03, 3.1623e-03, 2.3714e-03, 1.7783e-03, 1.3335e-03,
         1.0000e-03, 7.4989e-04, 5.6234e-04, 4.2170e-04, 3.1623e-04, 2.3714e-04,
         1.7783e-04, 1.3335e-04],
        [2.0000e+00, 1.4998e+00, 1.1247e+00, 8.4339e-01, 6.3246e-01, 4.7427e-01,
         3.5566e-01, 2.6670e-01, 2.0000e-

In [47]:
torch.arange(0, d_k, 2).float()

tensor([ 0.,  2.,  4.,  6.,  8., 10., 12., 14., 16., 18., 20., 22., 24., 26.,
        28., 30., 32., 34., 36., 38., 40., 42., 44., 46., 48., 50., 52., 54.,
        56., 58., 60., 62.])

In [74]:
torch.manual_seed(1234)
x = torch.rand(size=(4, 10, 64))

max_values = torch.max(x, dim=-1).values.unsqueeze(-1).expand_as(x)

(x - max_values).max(dim=-1)

torch.return_types.max(
values=tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]]),
indices=tensor([[24, 41, 47, 12, 56, 52, 27,  5,  7, 55],
        [55,  8, 50, 41,  1, 52, 41, 16, 55, 62],
        [62,  1, 20, 19, 24, 21,  4,  0, 52, 31],
        [ 5,  4, 16, 22,  4, 39, 50, 15, 51,  7]]))

In [128]:
torch.manual_seed(1234)
q = torch.rand(size=(4, 10, 64))
k = torch.rand(size=(4, 10, 64))
v = torch.rand(size=(4, 10, 32))

*rest, d_k = q.shape
presoftmax = einsum(
    q,
    k,
    "b ... seq_len_i d_k, b ... seq_len_j d_k -> b ... seq_len_i seq_len_j",
)

mask = torch.tril(torch.ones(10, 10)).bool()

(presoftmax / pow(d_k, 1 / 2))[0]

mask, presoftmax

(tensor([[ True, False, False, False, False, False, False, False, False, False],
         [ True,  True, False, False, False, False, False, False, False, False],
         [ True,  True,  True, False, False, False, False, False, False, False],
         [ True,  True,  True,  True, False, False, False, False, False, False],
         [ True,  True,  True,  True,  True, False, False, False, False, False],
         [ True,  True,  True,  True,  True,  True, False, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True, False, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True, False, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True, False],
         [ True,  True,  True,  True,  True,  True,  True,  True,  True,  True]]),
 tensor([[[15.7716, 16.5862, 17.4225, 16.4606, 14.7275, 18.7561, 15.7986,
           17.1233, 14.5781, 16.8426],
          [15.0803, 13.4610, 16.1601, 13.7027, 13.7940, 16.5608, 13.5388,
 

In [91]:
import math


In [92]:
premask = presoftmax / math.sqrt(d_k)

premask

tensor([[[1.9714, 2.0733, 2.1778, 2.0576, 1.8409, 2.3445, 1.9748, 2.1404,
          1.8223, 2.1053],
         [1.8850, 1.6826, 2.0200, 1.7128, 1.7242, 2.0701, 1.6924, 1.8133,
          1.8130, 1.9556],
         [2.1716, 2.0596, 2.3665, 2.0296, 1.9975, 2.3372, 2.2420, 2.0584,
          2.0849, 2.1004],
         [1.9110, 1.8700, 2.1658, 1.8601, 1.7019, 2.0328, 1.9295, 1.8076,
          1.6721, 2.0695],
         [2.3246, 2.2209, 2.4238, 2.2668, 2.0163, 2.4346, 2.4636, 2.0866,
          2.0797, 2.3689],
         [2.1857, 2.1622, 2.2639, 2.0762, 1.8535, 2.3409, 2.0513, 2.1495,
          2.1183, 2.0036],
         [1.9781, 1.9013, 2.0556, 1.9259, 1.7705, 2.2400, 2.0226, 2.0701,
          1.7630, 1.9328],
         [2.0683, 1.8581, 2.2600, 1.9656, 1.9554, 2.1910, 2.0379, 1.8916,
          1.9197, 2.2851],
         [1.9250, 1.9862, 2.1762, 1.8666, 1.8369, 2.1149, 2.0933, 1.9009,
          1.8661, 2.1989],
         [1.8617, 1.8074, 2.1786, 1.9476, 1.7324, 1.9553, 1.8564, 1.7850,
          1.7111,

In [112]:
att_weights = torch.softmax(
    torch.where(mask, premask, -torch.inf),
    dim=-1,
)

In [125]:
einsum_v = einsum(
    att_weights,
    v,
    "b ... seq_len_i seq_len_j, b ... seq_len_j d_v -> b ... seq_len_i d_v",
)

In [126]:
einsum_v[0][0]

tensor([0.1382, 0.0087, 0.0231, 0.8914, 0.2184, 0.1071, 0.1477, 0.3850, 0.2150,
        0.8429, 0.1647, 0.6820, 0.9716, 0.0792, 0.1500, 0.7913, 0.1466, 0.4232,
        0.1505, 0.0987, 0.8933, 0.0761, 0.5398, 0.6706, 0.0347, 0.5323, 0.4878,
        0.3560, 0.8594, 0.5754, 0.4050, 0.4858])

In [124]:
att_weights.shape, att_weights[0][0]

(torch.Size([4, 10, 10]), tensor([1., 0., 0., 0., 0., 0., 0., 0., 0., 0.]))

In [ ]:
import torch

batch_size, feature_size = 3, 5
weights = torch.randn(feature_size, requires_grad=True)


def model(feature_vec):
    # Very simple linear model with activation
    return feature_vec.dot(weights).relu()


examples = torch.randn(batch_size, feature_size)
result = torch.vmap(model)(examples)

In [2]:
result.shape

torch.Size([3])